# Model 2: Multilingual RAG (Hỗ trợ tiếng Việt)

File này sẽ tạo ra một Database Vector riêng biệt (`manga_chroma_db_vn`) sử dụng mô hình ngôn ngữ đa năng (`paraphrase-multilingual-mpnet-base-v2`).

**Mục tiêu:** Giúp Chatbot hiểu được câu hỏi Tiếng Việt và tìm kiếm truyện chính xác hơn.

In [1]:
!pip install pandas chromadb sentence-transformers tqdm

In [2]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
import os
import shutil
from tqdm import tqdm # Debug

# Load data
input_file = 'processed_manga.pkl'
if not os.path.exists(input_file):
    print(f"Loi: Khong tim thay file '{input_file}', can chay file EDA_Analyse.ipynb truoc.")
else:
    df = pd.read_pickle(input_file)
    
    initial_count = len(df)
    df.drop_duplicates(subset=['mal_id'], keep='first', inplace=True)
    cleaned_count = len(df)
    print(f"Da load file va xu ly trung lap.") 
    
    # Check if search_text is None
    if 'search_text' not in df.columns:
        df['search_text'] = df['title'] + " " + df['genres_str'] + " " + df['synopsis']
        
    print(f"Load file hoan tat. Bo du lieu gom {len(df)} dong.")

Da load file va xu ly trung lap.
Load file hoan tat. Bo du lieu gom 52247 dong.


In [3]:
# Init database

DB_PATH_VN = "./manga_chroma_db_vn"  
if os.path.exists(DB_PATH_VN):
    shutil.rmtree(DB_PATH_VN)

# Create persistent client
chroma_client = chromadb.PersistentClient(path=DB_PATH_VN)

# Create collection of vectors
collection = chroma_client.get_or_create_collection(name="manga_collection_vn")

In [4]:
# Only run this when recreate manga_chroma_db_vn
model = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')

# Data collection
# ChromaDB: IDs (mal_id), Embeddings, Metadatas, Documents (search_text)
documents = df['search_text'].tolist()
ids = [str(x) for x in df['mal_id'].tolist()]

# Prepare metadata
metadatas = []
for index, row in df.iterrows():
    meta = {
        "title": str(row['title']),
        "score": float(row['score']),
        "genres": str(row['genres_str']), # Genres: string
        "popularity_score": float(row['popularity_score']), 
        "url": str(row['url'])
    }
    metadatas.append(meta)

# Batch processing
batch_size = 256
total_docs = len(documents)

for i in tqdm(range(0, total_docs, batch_size)):
    end_idx = min(i + batch_size, total_docs)
    batch_docs = documents[i:end_idx]
    batch_ids = ids[i:end_idx]
    batch_metadatas = metadatas[i:end_idx]
    
    # Create vectors
    batch_embeddings = model.encode(batch_docs)
    
    # Storing
    collection.add(
        ids=batch_ids,
        embeddings=batch_embeddings,
        metadatas=batch_metadatas,
        documents=batch_docs
    )

print(f"\nStoring complete.")

100%|██████████| 205/205 [1:19:23<00:00, 23.24s/it]


Storing complete.


In [5]:
# Test

DB_PATH = "./manga_chroma_db_vn"
try:
    client = chromadb.PersistentClient(path=DB_PATH)
    collection = client.get_collection("manga_collection")
    print(f"Da ket noi thanh cong voi co so du lieu. Bo du lieu hien co {collection.count()} bo truyen.")
except Exception as e:
    print(f"Loi: Khong ket noi duoc voi co so du lieu. Hay chay lai mo hinh. {e}")

# Reload model 
model = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')

# Chatbot behaviour 
def test_query(text):
    print(f"\nQuery: {text}")
    vec = model.encode(text).tolist()
    res = collection.query(query_embeddings=[vec], n_results=3)
    
    for i, title in enumerate(res['metadatas'][0]):
        score = 1 - res['distances'][0][i]
        print(f"   #{i+1}: {title['title']} (Score: {score:.2f})")


Loi: Khong ket noi duoc voi co so du lieu. Hay chay lai mo hinh. Collection [manga_collection] does not exist


In [6]:
# Test prompt 
test_query("Truyện về hải tặc tìm kho báu")
test_query("Truyện tình cảm buồn học đường")
test_query("Thợ săn quái vật")


Query: Truyện về hải tặc tìm kho báu
   #1: One Piece Novel: A (Score: -3.55)
   #2: Captain Kid (Score: -3.69)
   #3: Oumu mo Kangaeteiru. (Score: -4.01)

Query: Truyện tình cảm buồn học đường
   #1: Zassou-tachi yo Taishi wo Idake (Score: -3.59)
   #2: Impatiens (Score: -4.21)
   #3: Douka Ore wo Houtteoite Kure: Nazeka Bocchi no Owatta Koukou Seikatsu wo Kanojo ga Kaeyou to Shitekuru (Score: -4.29)

Query: Thợ săn quái vật
   #1: Monster Hunter (Score: -2.70)
   #2: Monster Hunter: Senkou no Kariudo (Score: -2.77)
   #3: Teriyaki Western (Score: -2.79)
